### REGRAS

Piotroski
==============
Grupo 1: Rentabilidade (Profitability)
* ROA -> Se é positivo no ano corrente (1 ponto)
* ROA_-1 -> Se o ROA do ano corrente é maior que do ano anterior (1 ponto)
* FCO -> O Fluxo de Caixa Operacional é positivo no ano corrente? (1 ponto)
* Accurals (acumulação) -> O FCO é maior que o Lucro líquido ? (1 ponto)
 
 Grupo 2: Alavancagem e Financiamento (Leverage / Funding) (1 ponto para cada se sim)
* A dívida de longo prazo diminuiu em relação ao ano passado?; (longTermLoansAndFinancing, ou algum dado que mostre a divida de longo prazo) (ou seja, a dívida do ano corrente tem que ser menor que a divida do ano anterior)
* A liquidez corrente aumentou? (currentRatio, ou algum dado que mostre o valor da liq. corrente) (ou seja, a liq. do ano corrente tem que ser maior que a do ano anterior)
* O número de ações continuou igual ou diminuiu? (sharesOutstanding, ou outro dado que mostre o numero total de ações) (se o nº de ações do ano corrente for maior que do ano anterior, NÂO DAR 1 ponto) (diluição)

Grupo 3: Eficiência Operacional (Efficiency) (1 ponto se sim)
* A margem bruta aumentou? (grossMargins, ou outro dado que dê o valor da margem bruta) (ou seja, se a margem do ano corrente for maior que a do ano anterior) 
* O Giro do Ativo aumentou? (Giro de ativos = totalRevenue / totalAssets) ou outro dado que dê o valor de giro de ativo . (ou seja, se for maior que do ano anterior)

================================== 

In [1]:
import pandas as pd
import os

In [21]:
import pandas as pd
import os

anos = [str(a) for a in range(2014, 2026)]   # 2014 entra só como "ano anterior" de 2015

# nome da coluna no Comdinheiro -> nome curto
indicadores = {
    'ROA': 'ROA',
    'Fluxo de Caixa das Operações': 'FCO',
    'Lucro Líquido': 'LL',
    'LC': 'LC',
    'TOTAL_ACOES_EX_TESOURARIA': 'ACOES',
    'Margem Bruta(%)': 'MB',
    'Receita Líquida': 'RECEITA',
    'Ativo Total': 'ATIVO',
    'Empréstimos e Financiamentos - Longo Prazo': 'DIV_LP',
}

planilhas = []
for p in os.listdir('planilhas'):
    df = pd.read_excel('planilhas/' + p).set_index('Data')
    df.columns = [c.strip() for c in df.columns]   # tira os \n do final: "ALOS3\nLucro Líquido"
    planilhas.append(df)

# 1) uma tabela por ano: linha = ticker, colunas = indicadores
dados = {}
for ano in anos:
    dc = {}
    for df in planilhas:
        linha = df.loc[f'{ano}-12-31']
        tickers = [c.split('\n')[0] for c in df.columns if c.endswith('\nTICKER')]
        for ticker in tickers:
            dc[ticker] = {curto: linha[f'{ticker}\n{nome}'] for nome, curto in indicadores.items()}
    tabela = pd.DataFrame(dc).T.astype(float)
    tabela['GIRO'] = tabela['RECEITA'] / tabela['ATIVO']
    tabela['ALAV'] = tabela['DIV_LP'] / tabela['ATIVO']
    dados[ano] = tabela

# 2) compara cada ano com o anterior
for ano in anos[1:]:
    atual = dados[ano]
    ant = dados[str(int(ano) - 1)].reindex(atual.index)   # mesmo ticker na mesma linha

    f = pd.DataFrame(index=atual.index)
    # Rentabilidade
    f['ROA>0']     = atual['ROA'] > 0
    f['ROA_sub']   = atual['ROA'] > ant['ROA']
    f['FCO>0']     = atual['FCO'] > 0
    f['ACCRUAL']   = atual['FCO'] > atual['LL']
    # Alavancagem
    f['ALAV_cai']  = atual['ALAV'] < ant['ALAV']
    f['LC_sobe']   = atual['LC'] > ant['LC']
    f['SEM_DIL']   = atual['ACOES'] <= ant['ACOES']
    # Eficiência
    f['MB_sobe']   = atual['MB'] > ant['MB']
    f['GIRO_sobe'] = atual['GIRO'] > ant['GIRO']

    f = f.astype(int)                 # True/False -> 1/0
    f['PIOTROSKI'] = f.sum(axis=1)

    # pio_min = f['PIOTROSKI'].min()
    # pio_max = f['PIOTROSKI'].max()

    f['p_norm'] = (f['PIOTROSKI']-1)/(8)

    # só pontua quem tem dado completo nos dois anos
    completo = atual.notna().all(axis=1) & ant.notna().all(axis=1)
    f = f[completo]
    f.sort_values('PIOTROSKI', ascending=False).to_excel(f'resultado/{ano}.xlsx')

In [8]:
df = pd.read_excel('resultado/2015.xlsx')['PIOTROSKI']

In [19]:
f = pd.DataFrame(index=df.index)
f['total_p'] = (df-df.min())/(df.max()-df.min())
f

,total_p
0,1.000000
1,1.000000
2,1.000000
3,1.000000
4,1.000000
...,...
59,0.166667
60,0.166667
61,0.166667
62,0.000000
